In [1]:
import pandas as pd


In [2]:
# Load cleaned dataset
df = pd.read_csv("HHS_Cleaned_Daily_Data.csv")

In [3]:
# Convert Date column to datetime
df["Date"] = pd.to_datetime(df["Date"])

## 1. Identify Missing Dates

In [4]:
all_dates = pd.date_range(df["Date"].min(), df["Date"].max(), freq="D")

missing_dates = all_dates.difference(df["Date"])

print("========== Missing Dates ==========")
print(missing_dates)

========== Missing Dates ==========
DatetimeIndex([], dtype='datetime64[us]', freq='D')


## 2. Identify Duplicate Dates

In [5]:
duplicate_dates = df[df["Date"].duplicated(keep=False)]

print("\n========== Duplicate Dates ==========")
print(duplicate_dates)


========== Duplicate Dates ==========
Empty DataFrame
Columns: [Date, Children apprehended and placed in CBP custody*, Children in CBP custody, Children transferred out of CBP custody, Children in HHS Care, Children discharged from HHS Care, Validation]
Index: []


## 3. Validate Logical Constraints

In [8]:
print(df[[
    "Children transferred out of CBP custody",
    "Children in CBP custody",
    "Children discharged from HHS Care",
    "Children in HHS Care"
]].dtypes)

Children transferred out of CBP custody    float64
Children in CBP custody                    float64
Children discharged from HHS Care          float64
Children in HHS Care                           str
dtype: object


In [9]:
print(df["Children in HHS Care"].head(20))

0     7,903
1     7,903
2     7,903
3     8,492
4     7,002
5     7,002
6     8,103
7     8,103
8     8,103
9     9,250
10    9,250
11    9,250
12    9,250
13    9,250
14    9,250
15    9,250
16    9,250
17    9,250
18    9,250
19    9,250
Name: Children in HHS Care, dtype: str


In [10]:
df["Children in HHS Care"] = (
    df["Children in HHS Care"]
        .astype(str)
        .str.replace(",", "", regex=False)   # Remove commas
        .str.strip()                          # Remove extra spaces
        .replace({"": None, "NA": None, "N/A": None, "-": None})
)

df["Children in HHS Care"] = pd.to_numeric(
    df["Children in HHS Care"],
    errors="coerce"
)

In [11]:
# Transfers should not exceed CBP Custody
df["Transfers_Error"] = df["Children transferred out of CBP custody"] > df["Children in CBP custody"]

# Discharges should not exceed HHS Care
df["Discharges_Error"] = df["Children discharged from HHS Care"] > df["Children in HHS Care"]

## 4. Flag Reporting Anomalies

In [12]:
df["Reporting_Anomaly"] = (
    df["Transfers_Error"] |
    df["Discharges_Error"]
)

# Display anomalies
anomalies = df[df["Reporting_Anomaly"]]

print("\n========== Reporting Anomalies ==========")
print(anomalies)


========== Reporting Anomalies ==========
Empty DataFrame
Columns: [Date, Children apprehended and placed in CBP custody*, Children in CBP custody, Children transferred out of CBP custody, Children in HHS Care, Children discharged from HHS Care, Validation, Transfers_Error, Discharges_Error, Reporting_Anomaly]
Index: []


## 5. Summary Report

In [13]:

print("\n========== Data Quality Summary ==========")
print(f"Total Records            : {len(df)}")
print(f"Missing Dates            : {len(missing_dates)}")
print(f"Duplicate Date Records   : {len(duplicate_dates)}")
print(f"Transfer Violations      : {df['Transfers_Error'].sum()}")
print(f"Discharge Violations     : {df['Discharges_Error'].sum()}")
print(f"Total Reporting Anomalies: {df['Reporting_Anomaly'].sum()}")


========== Data Quality Summary ==========
Total Records            : 1085
Missing Dates            : 0
Duplicate Date Records   : 0
Transfer Violations      : 0
Discharge Violations     : 0
Total Reporting Anomalies: 0


## 6. Save Dataset with Validation Flags

In [14]:
df.to_csv("HHS_Validated_Data.csv", index=False)

print("\nValidation completed successfully.")


Validation completed successfully.
